In [1]:
%matplotlib widget
import inspect
import re
def debugPrint(x):
    frame = inspect.currentframe().f_back
    s = inspect.getframeinfo(frame).code_context[0]
    r = re.search(r"\((.*)\)", s).group(1)
    print("{} [{}] = {}".format(r,type(x).__name__, x))
       
import torch
import numpy as np
import warp as wp

# Initialize Warp
wp.config.verify_autograd_array_access = False
wp.config.verbose = False
wp.init()

from sphWarpCore import radiusSearchCompactHashMap, sphOperation_warp
from sphWarpCore.enumTypes import *

from diffSPH.neighborhood import buildNeighborhood
from diffSPH.operations import SPHOperation
from diffSPH.enums import Operation, SupportScheme, GradientMode, LaplacianMode
from diffSPH.operations import KernelCorrectionScheme
from diffSPH.kernels import *
from diffSPH.plotting import visualizeParticles
from diffSPH.kernels import getSPHKernelv2

import matplotlib.pyplot as plt
from demo_util import *

Warp 1.12.0 initialized:
   CUDA Toolkit 12.9, Driver 13.2
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA RTX PRO 500 Blackwell Generation Laptop GPU" (6 GiB, sm_120, mempool enabled)
   Kernel cache:
     /home/lu26029/.cache/warp/1.12.0


In [2]:
device = torch.device('cpu')
device = torch.device('cuda')
targetNumNeighbors = 50
nx = 128
dim = 2
numParticles = nx**dim

warpOnly = False
periodic = True

particleState, domain, adjacency, neighborhood, simulationState, measurements = prepData(nx, targetNumNeighbors, dim, device, periodic, warpOnly)

Module sphWarpCore.radiusSearch.wp_compactHash e2c9126 load on device 'cuda:0' took 2.05 ms  (cached)
Module sphWarpCore.operations.wp_density b70beeb load on device 'cuda:0' took 2.36 ms  (cached)


In [3]:
from sphWarpCore.radiusSearch.verlet import *

adjacencyV = buildVerletList(
    particleState,
    domain, verletScale = 2**(1/dim),
    supportMode = SupportScheme.SuperSymmetric,
    priorNeighborhood = None,
    verbose = True
)
adjacencyV = buildVerletList(
    particleState,
    domain, verletScale = 2**(1/dim),
    supportMode = SupportScheme.SuperSymmetric,
    priorNeighborhood = adjacencyV,
    verbose = True
)

filteredAdjacencyV = filterVerletList(
    particleState,
    domain, adjacencyV,
    supportMode = SupportScheme.SuperSymmetric
)



Building neighborhood from scratch [no prior neighborhood]
Distance a: min: 0.0, max: 0.0, avg: 0.0
Distance b: min: 0.0, max: 0.0, avg: 0.0
Support a: min: 0.0881546214222908, max: 0.0881546214222908, avg: 0.0881546214222908
Support b: min: 0.0881546214222908, max: 0.0881546214222908, avg: 0.0881546214222908
Support Factor: 0.025819890201091766, Minimum Support: 0.0881546214222908
Max Distance: 0.0
Distance Factor: 0.0
Reusing neighborhood
Module sphWarpCore.radiusSearch.verlet eaac164 load on device 'cuda:0' took 1.09 ms  (cached)
Module sphWarpCore.radiusSearch.verlet 8454079 load on device 'cuda:0' took 1.27 ms  (cached)


In [4]:
print(filteredAdjacencyV.i.shape, adjacencyV.i.shape, adjacency.i.shape)

print(torch.all(filteredAdjacencyV.i == adjacency.i))

torch.Size([791762]) torch.Size([1626316]) torch.Size([791762])
tensor(True, device='cuda:0')
